# E1

## 1. Lectura y exploración

In [1]:
import pandas as pd
import sqlite3

ruta_csv = "car_prices.csv"
df = pd.read_csv(ruta_csv)

In [2]:
print(f"Dimensiones: {df.shape}")
print(f"Primeras filas:\n{df.head()}")

Dimensiones: (71779, 16)
Primeras filas:
   year    make  model          trim   body transmission                vin  \
0  2014     Kia  Forte            LX  Sedan    automatic  knafx4a64e5093418   
1  2010    Audi     A4  2.0T Premium  Sedan    automatic  wauafafl4an066787   
2  2011  Toyota  Camry            LE  Sedan    automatic  4t1bf3ek3bu171593   
3  2007     NaN    NaN           NaN    NaN    automatic  5tfrv54117x028139   
4  2014     Kia  Forte            LX  Sedan          NaN  knafx4a66e5208858   

  state  condition  odometer   color interior                    seller  \
0    nc       29.0   38339.0     red    black          avis corporation   
1    fl       46.0   46120.0  silver    black       lauderdale infiniti   
2    ga       19.0   55958.0    gray     gray  capital one auto finance   
3    ca       25.0   67044.0    gray     gray        e/e auto wholesale   
4    ga       46.0   10386.0  silver    black   kia motors america  inc   

       mmr  sellingprice         

In [3]:
print(f"Resumen de información:\n{df.info()}")

<class 'pandas.DataFrame'>
RangeIndex: 71779 entries, 0 to 71778
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   year          71779 non-null  int64  
 1   make          71178 non-null  str    
 2   model         71166 non-null  str    
 3   trim          71027 non-null  str    
 4   body          69869 non-null  str    
 5   transmission  63135 non-null  str    
 6   vin           71779 non-null  str    
 7   state         71779 non-null  str    
 8   condition     66451 non-null  float64
 9   odometer      71751 non-null  float64
 10  color         71662 non-null  str    
 11  interior      71662 non-null  str    
 12  seller        71779 non-null  str    
 13  mmr           71779 non-null  float64
 14  sellingprice  71779 non-null  float64
 15  saledate      71779 non-null  str    
dtypes: float64(4), int64(1), str(11)
memory usage: 8.8 MB
Resumen de información:
None


In [4]:
print(f"Estadísticas descriptivas:\n{df.describe()}")
print("\nValores nulos por columna:")
print(df.isna().sum().to_string())

Estadísticas descriptivas:
               year     condition       odometer            mmr   sellingprice
count  71779.000000  66451.000000   71751.000000   71779.000000   71779.000000
mean    2009.547611     29.877398   72181.653399   13007.860586   12727.151158
std        4.167479     13.789741   56288.561615    9393.949324    9438.339868
min     1984.000000      1.000000       1.000000      25.000000       1.000000
25%     2007.000000     22.000000   28526.000000    6025.000000    5700.000000
50%     2011.000000     34.000000   56447.000000   11750.000000   11500.000000
75%     2013.000000     41.000000  106286.000000   17500.000000   17400.000000
max     2015.000000     49.000000  999999.000000  176000.000000  169500.000000

Valores nulos por columna:
year               0
make             601
model            613
trim             752
body            1910
transmission    8644
vin                0
state              0
condition       5328
odometer          28
color            117
int

Las columnas describen año, fabricante, modelo, versión, carrocería, transmisión, VIN, estado, condición, odómetro, colores, vendedor, valor de referencia (MMR - Manheim Market Report), precio de venta y fecha. `year`, `condition`, `odometer`, `mmr` y `sellingprice` son numéricas y los demás campos son texto. 

Hay faltantes en `make`, `model`, `trim`, `body`, `transmission`, `condition`, `odometer`, `color` e `interior`.

`vin` identifica un vehículo y es la llave candidata (`NOT NULL`, `UNIQUE`). Como se repite en registros de ventas, el vehículo se almacena una sola vez y cada transacción se guarda en `Ventas`. 

Las llaves foráneas conectan vehículo con modelo y carrocería, modelo con fabricante, y venta con vehículo y vendedor.



## 2. Modelo relacional

- `Fabricantes(fid, nombre)` 1:N `Modelos`.
- `Modelos(mid, nombre, fabricante_id)` 1:N `Vehiculos`.
- `Carrocerias(cid, tipo)` 1:N `Vehiculos`.
- `Vehiculos(veid, vin, year, modelo_id, trim, carroceria_id, transmission, color, interior)` 1:N `Ventas`.
- `Vendedores(vid, nombre)` 1:N `Ventas`.
- `Ventas(sid, vehiculo_id, vendedor_id, state, condition, odometer, mmr, sellingprice, saledate)`.

Separar estas entidades evita repetir nombres de fabricantes, modelos, carrocerías y vendedores en cada transacción. 

Los atributos variables o de la transacción quedan en su entidad correspondiente.

## 3. Creación

Se crea `vehicles_e1.db` para conservar la base de datos.


Los valores faltantes opcionales se guardan como `NULL`, se omiten filas sin VIN, fabricante, modelo o vendedor porque son necesarios para identificar y relacionar los registros.

In [ ]:
ruta_db = "vehicles_e1.db"
conn = sqlite3.connect(ruta_db)

cur = conn.cursor()

cur.execute("PRAGMA foreign_keys = ON")

cur.executescript("""
DROP TABLE IF EXISTS Ventas;
DROP TABLE IF EXISTS Vehiculos;
DROP TABLE IF EXISTS Vendedores;
DROP TABLE IF EXISTS Carrocerias;
DROP TABLE IF EXISTS Modelos;
DROP TABLE IF EXISTS Fabricantes;

CREATE TABLE Fabricantes (
    fid INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL UNIQUE
);
CREATE TABLE Modelos (
    mid INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL,
    fabricante_id INTEGER NOT NULL,
    UNIQUE (nombre, fabricante_id),
    FOREIGN KEY (fabricante_id) REFERENCES Fabricantes(fid)
);
CREATE TABLE Carrocerias (
    cid INTEGER PRIMARY KEY,
    tipo TEXT NOT NULL UNIQUE
);
CREATE TABLE Vendedores (
    vid INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL UNIQUE
);
CREATE TABLE Vehiculos (
    veid INTEGER PRIMARY KEY,
    vin TEXT NOT NULL UNIQUE,
    year INTEGER,
    modelo_id INTEGER NOT NULL,
    trim TEXT,
    carroceria_id INTEGER,
    transmission TEXT,
    color TEXT,
    interior TEXT,
    FOREIGN KEY (modelo_id) REFERENCES Modelos(mid),
    FOREIGN KEY (carroceria_id) REFERENCES Carrocerias(cid)
);
CREATE TABLE Ventas (
    sid INTEGER PRIMARY KEY,
    vehiculo_id INTEGER NOT NULL,
    vendedor_id INTEGER NOT NULL,
    state TEXT,
    condition REAL,
    odometer REAL,
    mmr REAL,
    sellingprice REAL,
    saledate TEXT,
    FOREIGN KEY (vehiculo_id) REFERENCES Vehiculos(veid),
    FOREIGN KEY (vendedor_id) REFERENCES Vendedores(vid)
);
""")

In [6]:
def valor(x):
    return None if pd.isna(x) else x.item() if hasattr(x, "item") else x

for nombre in df["make"].dropna().unique():
    cur.execute("INSERT INTO Fabricantes(nombre) VALUES (?)", (nombre,))

for nombre in df[["model", "make"]].dropna().drop_duplicates().itertuples(index=False, name=None):
    make_id = cur.execute("SELECT fid FROM Fabricantes WHERE nombre = ?", (nombre[1],)).fetchone()[0]
    cur.execute("INSERT INTO Modelos(nombre, fabricante_id) VALUES (?, ?)", (nombre[0], make_id))

for tipo in df["body"].dropna().unique():
    cur.execute("INSERT INTO Carrocerias(tipo) VALUES (?)", (tipo,))

for nombre in df["seller"].dropna().unique():
    cur.execute("INSERT INTO Vendedores(nombre) VALUES (?)", (nombre,))



In [7]:
fabricantes = dict(cur.execute("SELECT nombre, fid FROM Fabricantes"))

modelos = {(nombre, fabricantes_nombre): mid for mid, nombre, fabricantes_nombre in cur.execute(
    "SELECT Modelos.mid, Modelos.nombre, Fabricantes.nombre FROM Modelos JOIN Fabricantes ON Modelos.fabricante_id = Fabricantes.fid")}

carrocerias = dict(cur.execute("SELECT tipo, cid FROM Carrocerias"))

vendedores = dict(cur.execute("SELECT nombre, vid FROM Vendedores"))

In [8]:
fabricantes

{'Kia': 1,
 'Audi': 2,
 'Toyota': 3,
 'Volvo': 4,
 'Dodge': 5,
 'Ford': 6,
 'Chevrolet': 7,
 'Honda': 8,
 'Buick': 9,
 'Mercury': 10,
 'BMW': 11,
 'Jeep': 12,
 'Mercedes-Benz': 13,
 'Cadillac': 14,
 'Lincoln': 15,
 'Nissan': 16,
 'chevrolet': 17,
 'Jaguar': 18,
 'Infiniti': 19,
 'Ram': 20,
 'Mazda': 21,
 'Lexus': 22,
 'Hyundai': 23,
 'GMC': 24,
 'Volkswagen': 25,
 'Suzuki': 26,
 'Saturn': 27,
 'Chrysler': 28,
 'dodge': 29,
 'Acura': 30,
 'Mitsubishi': 31,
 'Subaru': 32,
 'Bentley': 33,
 'FIAT': 34,
 'Scion': 35,
 'Pontiac': 36,
 'mercury': 37,
 'MINI': 38,
 'Land Rover': 39,
 'Porsche': 40,
 'Saab': 41,
 'smart': 42,
 'hyundai': 43,
 'ford': 44,
 'mazda': 45,
 'HUMMER': 46,
 'Oldsmobile': 47,
 'mitsubishi': 48,
 'Plymouth': 49,
 'lexus': 50,
 'volkswagen': 51,
 'gmc': 52,
 'nissan': 53,
 'cadillac': 54,
 'chrysler': 55,
 'honda': 56,
 'porsche': 57,
 'bmw': 58,
 'Maserati': 59,
 'land rover': 60,
 'toyota': 61,
 'lincoln': 62,
 'pontiac': 63,
 'mercedes': 64,
 'acura': 65,
 'jeep': 66,

In [9]:
modelos

{('Forte', 'Kia'): 1,
 ('A4', 'Audi'): 2,
 ('Camry', 'Toyota'): 3,
 ('XC90', 'Volvo'): 4,
 ('Grand Caravan', 'Dodge'): 5,
 ('F-150', 'Ford'): 6,
 ('Cruze', 'Chevrolet'): 7,
 ('Avenger', 'Dodge'): 8,
 ('Civic', 'Honda'): 9,
 ('Focus', 'Ford'): 10,
 ('Enclave', 'Buick'): 11,
 ('Optima', 'Kia'): 12,
 ('Taurus', 'Ford'): 13,
 ('Tahoe', 'Chevrolet'): 14,
 ('Mountaineer', 'Mercury'): 15,
 ('Malibu', 'Chevrolet'): 16,
 ('F-350 Super Duty', 'Ford'): 17,
 ('3 Series', 'BMW'): 18,
 ('Cherokee', 'Jeep'): 19,
 ('CR-V', 'Honda'): 20,
 ('E-Class', 'Mercedes-Benz'): 21,
 ('Odyssey', 'Honda'): 22,
 ('Escalade', 'Cadillac'): 23,
 ('MKS', 'Lincoln'): 24,
 ('Corolla', 'Toyota'): 25,
 ('Sorento', 'Kia'): 26,
 ('Leaf', 'Nissan'): 27,
 ('Sienna', 'Toyota'): 28,
 ('1500', 'chevrolet'): 29,
 ('MKX', 'Lincoln'): 30,
 ('Accord', 'Honda'): 31,
 ('Impala', 'Chevrolet'): 32,
 ('Altima', 'Nissan'): 33,
 ('XJ', 'Jaguar'): 34,
 ('Highlander', 'Toyota'): 35,
 ('Express', 'Chevrolet'): 36,
 ('G Sedan', 'Infiniti'): 37,

In [10]:
carrocerias

{'Sedan': 1,
 'SUV': 2,
 'Minivan': 3,
 'Regular Cab': 4,
 'Hatchback': 5,
 'Crew Cab': 6,
 'Van': 7,
 'G Sedan': 8,
 'Coupe': 9,
 'Extended Cab': 10,
 'Quad Cab': 11,
 'Wagon': 12,
 'SuperCab': 13,
 'SuperCrew': 14,
 'Convertible': 15,
 'Double Cab': 16,
 'E-Series Van': 17,
 'King Cab': 18,
 'G Coupe': 19,
 'G Convertible': 20,
 'Xtracab': 21,
 'sedan': 22,
 'Club Cab': 23,
 'CTS Coupe': 24,
 'CrewMax Cab': 25,
 'Access Cab': 26,
 'Koup': 27,
 'Beetle Convertible': 28,
 'Genesis Coupe': 29,
 'Elantra Coupe': 30,
 'TSX Sport Wagon': 31,
 'G37 Convertible': 32,
 'Q60 Coupe': 33,
 'Cab Plus 4': 34,
 'Cab Plus': 35,
 'Mega Cab': 36,
 'Promaster Cargo Van': 37,
 'CTS-V Wagon': 38,
 'Q60 Convertible': 39,
 'Ram Van': 40,
 'CTS Wagon': 41,
 'CTS-V Coupe': 42,
 'G37 Coupe': 43,
 'convertible': 44}

In [11]:
vendedores

{'avis corporation': 1,
 'lauderdale infiniti': 2,
 'capital one auto finance': 3,
 'e/e auto wholesale': 4,
 'kia motors america  inc': 5,
 'remarketing by ge/hardy automotive llc': 6,
 'performance chrysler jeep dodge inc': 7,
 'u-haul': 8,
 'gm remarketing': 9,
 'chrysler capital': 10,
 'ahfc/honda lease trust/hvt  inc. eot': 11,
 'ford motor credit company llc': 12,
 'bmw of san antonio': 13,
 'remarketing by ge/space coast credit union': 14,
 'avis budget group': 15,
 'young motors llc': 16,
 'ed morse auto plaza': 17,
 'gm financial leasing': 18,
 'remarketing by ge/budget motor co': 19,
 'quality acceptance': 20,
 'sterling credit corp': 21,
 'the hertz corporation': 22,
 'chrysler group/hertz/pv holding/000gd': 23,
 'boch toyota/scion south': 24,
 'mercedes-benz financial services': 25,
 'primeritus remarketing/aas': 26,
 'daylight auto financing inc': 27,
 'ars/avis budget group': 28,
 'ford motor credit company': 29,
 'boch new to you superstore': 30,
 'nissan-infiniti lt': 3

# 4. Carga

In [12]:
vehiculos = {}

for r in df.drop_duplicates("vin").itertuples(index=False, name=None):
    (year, make, model, trim, body, transmission, vin, state, condition, odometer,
     color, interior, seller, mmr, sellingprice, saledate) = r
    if pd.isna(vin) or pd.isna(make) or pd.isna(model):
        continue
    cur.execute("""INSERT INTO Vehiculos
        (vin, year, modelo_id, trim, carroceria_id, transmission, color, interior)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)""",
        (valor(vin), valor(year), modelos[(model, make)], valor(trim),
         carrocerias.get(body) if not pd.isna(body) else None,
         valor(transmission), valor(color), valor(interior)))
    vehiculos[vin] = cur.lastrowid

for r in df.itertuples(index=False, name=None):
    (year, make, model, trim, body, transmission, vin, state, condition, odometer,
     color, interior, seller, mmr, sellingprice, saledate) = r
    if pd.isna(vin) or pd.isna(make) or pd.isna(model) or pd.isna(seller) or vin not in vehiculos:
        continue
    cur.execute("""INSERT INTO Ventas
        (vehiculo_id, vendedor_id, state, condition, odometer, mmr, sellingprice, saledate)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)""",
        (vehiculos[vin], vendedores[seller], valor(state), valor(condition),
         valor(odometer), valor(mmr), valor(sellingprice), valor(saledate)))

conn.commit()

In [13]:
print("Registros por tabla:")
for tabla in ("Fabricantes", "Modelos", "Carrocerias", "Vendedores", "Vehiculos", "Ventas"):
    print(f"{tabla}: {cur.execute(f'SELECT COUNT(*) FROM {tabla}').fetchone()[0]}")

Registros por tabla:
Fabricantes: 91
Modelos: 872
Carrocerias: 44
Vendedores: 6064
Vehiculos: 70967
Ventas: 71166


## 5. Verificación de integridad

In [14]:
print("Tablas y columnas:")
for (tabla,) in cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall():
    print(f"\n{tabla}")
    print(cur.execute(f"PRAGMA table_info([{tabla}])").fetchall())

try:
    cur.execute("INSERT INTO Ventas (vehiculo_id, vendedor_id) VALUES (?, ?)", (999999999, 1))
    conn.commit()
except sqlite3.IntegrityError as error:
    conn.rollback()
    print("\nInserción inválida rechazada por la llave foránea:", error)

Tablas y columnas:

Carrocerias
[(0, 'cid', 'INTEGER', 0, None, 1), (1, 'tipo', 'TEXT', 1, None, 0)]

Fabricantes
[(0, 'fid', 'INTEGER', 0, None, 1), (1, 'nombre', 'TEXT', 1, None, 0)]

Modelos
[(0, 'mid', 'INTEGER', 0, None, 1), (1, 'nombre', 'TEXT', 1, None, 0), (2, 'fabricante_id', 'INTEGER', 1, None, 0)]

Vehiculos
[(0, 'veid', 'INTEGER', 0, None, 1), (1, 'vin', 'TEXT', 1, None, 0), (2, 'year', 'INTEGER', 0, None, 0), (3, 'modelo_id', 'INTEGER', 1, None, 0), (4, 'trim', 'TEXT', 0, None, 0), (5, 'carroceria_id', 'INTEGER', 0, None, 0), (6, 'transmission', 'TEXT', 0, None, 0), (7, 'color', 'TEXT', 0, None, 0), (8, 'interior', 'TEXT', 0, None, 0)]

Vendedores
[(0, 'vid', 'INTEGER', 0, None, 1), (1, 'nombre', 'TEXT', 1, None, 0)]

Ventas
[(0, 'sid', 'INTEGER', 0, None, 1), (1, 'vehiculo_id', 'INTEGER', 1, None, 0), (2, 'vendedor_id', 'INTEGER', 1, None, 0), (3, 'state', 'TEXT', 0, None, 0), (4, 'condition', 'REAL', 0, None, 0), (5, 'odometer', 'REAL', 0, None, 0), (6, 'mmr', 'REAL', 0,

In [15]:
print("\nVentas con fabricante:")
cur.execute("""SELECT Ventas.sid, Vehiculos.vin, Fabricantes.nombre AS fabricante,
                  Ventas.sellingprice
           FROM Ventas
           JOIN Vehiculos ON Ventas.vehiculo_id = Vehiculos.veid
           JOIN Modelos ON Vehiculos.modelo_id = Modelos.mid
           JOIN Fabricantes ON Modelos.fabricante_id = Fabricantes.fid
           LIMIT 5""")
for fila in cur.fetchall():
    print(fila)
conn.close()


Ventas con fabricante:
(1, 'knafx4a64e5093418', 'Kia', 11500.0)
(2, 'wauafafl4an066787', 'Audi', 16600.0)
(3, '4t1bf3ek3bu171593', 'Toyota', 9000.0)
(4, 'knafx4a66e5208858', 'Kia', 12800.0)
(5, 'yv4cy982881434167', 'Volvo', 11300.0)
